[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/llm-course/blob/main/slides/week9/thinking_demo.ipynb)

# The thinking revolution — hands-on exploration

**PSYC 51.17: Models of language and communication**  
**Week 9 — Companion to Lecture 24**

---

## Learning objectives

By the end of this session, you will:
1. Implement chain-of-thought prompting and compare it to standard prompting
2. Use Claude's extended thinking API to observe *visible* reasoning traces
3. Measure how thinking budget affects answer quality on math problems
4. Implement a simplified GRPO training loop to see how RL produces reasoning
5. Explore the "budget forcing" trick from the s1 paper

## Setup

In [ ]:
# Install required packages (for Colab)
!pip install -q anthropic torch matplotlib numpy

In [ ]:
import os
import time
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("\u2713 All imports successful!")

## Part 1: Chain-of-thought prompting

Before reasoning models existed, [Wei et al. (2022)](https://arxiv.org/abs/2201.11903) discovered that simply asking a model to **show its work** dramatically improves accuracy. Let's see this in action.

### Setting up the Anthropic client

You'll need an API key from [console.anthropic.com](https://console.anthropic.com). Set it as an environment variable or paste it below.

In [ ]:
from google.colab import userdata

# Try to get the key from Colab secrets first, then fall back to environment variable
try:
    api_key = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    api_key = os.environ.get('ANTHROPIC_API_KEY', '')

if not api_key:
    api_key = input("Enter your Anthropic API key: ")

import anthropic
client = anthropic.Anthropic(api_key=api_key)
print("\u2713 Anthropic client ready!")

### Standard prompting vs. chain-of-thought

Let's test the same math problems with and without chain-of-thought prompting. We use a small, fast model (Haiku) to clearly see the difference — larger models sometimes reason internally even without prompting.

In [ ]:
# Math problems that benefit from step-by-step reasoning
problems = [
    {
        "question": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?",
        "answer": 11
    },
    {
        "question": "A store has 312 apples. They sell 147 apples in the morning and receive a shipment of 89 apples in the afternoon. How many apples do they have at the end of the day?",
        "answer": 254
    },
    {
        "question": "If a train travels at 60 mph for 2.5 hours, then at 80 mph for 1.5 hours, what is the total distance traveled?",
        "answer": 270
    },
    {
        "question": "A rectangular garden is 3 times as long as it is wide. If the perimeter is 96 meters, what is the area of the garden in square meters?",
        "answer": 432
    },
    {
        "question": "In a class of 40 students, 25 play soccer, 20 play basketball, and 10 play both. How many students play neither sport?",
        "answer": 5
    }
]


def ask_standard(question):
    """Standard prompting: just ask for the answer."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=100,
        messages=[{"role": "user", "content": f"{question}\n\nAnswer with just the number."}]
    )
    return response.content[0].text


def ask_cot(question):
    """Chain-of-thought prompting: ask the model to think step by step."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=[{"role": "user", "content": f"{question}\n\nLet's think step by step, then give the final numerical answer."}]
    )
    return response.content[0].text


print("=" * 70)
print("STANDARD PROMPTING vs. CHAIN-OF-THOUGHT")
print("=" * 70)

for i, problem in enumerate(problems):
    print(f"\n--- Problem {i+1} (correct answer: {problem['answer']}) ---")
    print(f"Q: {problem['question'][:80]}...")
    
    standard_answer = ask_standard(problem['question'])
    print(f"\nStandard: {standard_answer.strip()[:80]}")
    
    cot_answer = ask_cot(problem['question'])
    print(f"\nCoT: {cot_answer.strip()[:200]}")
    print()

### 💡 Discussion

- Did chain-of-thought improve accuracy on any of the problems? Which ones?
- Notice how the CoT responses are longer. Each token is a computation step — the model is literally performing more computation.
- Why might a small model (Haiku) benefit more from CoT than a large model (Opus)?
- Is there a downside to always using CoT prompting? (Hint: cost, latency)

## Part 2: Extended thinking — visible reasoning traces

Reasoning models go beyond CoT *prompting*: they are **trained via RL** to generate internal reasoning. Claude's extended thinking lets you see this reasoning directly.

Let's compare the same problem with different thinking budgets.

In [ ]:
def ask_with_thinking(question, budget_tokens=4000, model="claude-sonnet-4-20250514"):
    """Ask with extended thinking enabled, return thinking + answer + token counts."""
    start = time.time()
    response = client.messages.create(
        model=model,
        max_tokens=budget_tokens + 2000,  # budget + room for answer
        thinking={"type": "enabled", "budget_tokens": budget_tokens},
        messages=[{"role": "user", "content": question}]
    )
    elapsed = time.time() - start
    
    thinking_text = ""
    answer_text = ""
    thinking_tokens = 0
    
    for block in response.content:
        if block.type == "thinking":
            thinking_text = block.thinking
        elif block.type == "text":
            answer_text = block.text
    
    thinking_tokens = response.usage.cache_read_input_tokens if hasattr(response.usage, 'cache_read_input_tokens') else 0
    # Get actual thinking token usage from the response
    output_tokens = response.usage.output_tokens
    
    return {
        "thinking": thinking_text,
        "answer": answer_text,
        "output_tokens": output_tokens,
        "elapsed": elapsed,
        "thinking_length": len(thinking_text),
    }


# A challenging math problem
hard_problem = """Find all positive integers n such that n^2 + 2n + 4 is divisible by 7."""

print(f"Problem: {hard_problem}")
print("=" * 70)

# Compare different thinking budgets
budgets = [1024, 4096, 10000]
results = {}

for budget in budgets:
    print(f"\n--- Budget: {budget:,} tokens ---")
    result = ask_with_thinking(hard_problem, budget_tokens=budget)
    results[budget] = result
    
    print(f"Time: {result['elapsed']:.1f}s")
    print(f"Output tokens: {result['output_tokens']:,}")
    print(f"Thinking length: {result['thinking_length']:,} chars")
    print(f"\nThinking (first 300 chars):")
    print(f"  {result['thinking'][:300]}...")
    print(f"\nAnswer (first 200 chars):")
    print(f"  {result['answer'][:200]}")

In [ ]:
# Visualize the relationship between thinking budget and response characteristics
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

budget_labels = [f"{b//1000}K" for b in budgets]
thinking_lengths = [results[b]["thinking_length"] for b in budgets]
times = [results[b]["elapsed"] for b in budgets]
output_tokens = [results[b]["output_tokens"] for b in budgets]

axes[0].bar(budget_labels, thinking_lengths, color="#f39c12", alpha=0.8)
axes[0].set_title("Thinking Length (chars)", fontsize=13)
axes[0].set_xlabel("Budget")
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(budget_labels, times, color="#0984e3", alpha=0.8)
axes[1].set_title("Response Time (seconds)", fontsize=13)
axes[1].set_xlabel("Budget")
axes[1].grid(axis="y", alpha=0.3)

axes[2].bar(budget_labels, output_tokens, color="#00b894", alpha=0.8)
axes[2].set_title("Total Output Tokens", fontsize=13)
axes[2].set_xlabel("Budget")
axes[2].grid(axis="y", alpha=0.3)

plt.suptitle("How Thinking Budget Affects Model Behavior", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Examining the thinking trace

Let's look at the full thinking trace for the highest budget. Notice the structure: the model breaks the problem down, tries approaches, and sometimes backtracks.

In [ ]:
# Print the full thinking trace from the largest budget
max_budget = max(budgets)
thinking = results[max_budget]["thinking"]

print(f"Full thinking trace ({len(thinking):,} characters):")
print("=" * 70)
print(thinking)
print("=" * 70)
print(f"\nFinal answer:")
print(results[max_budget]["answer"])

### 💡 Discussion

- Does the model use more thinking when given a higher budget? Or does it use only what it needs?
- Look at the thinking trace: can you identify moments of **self-verification**, **backtracking**, or **alternative approaches**?
- These reasoning strategies *emerged from RL training*. The model was never explicitly taught to "check its work" — it learned that checking leads to more correct answers.
- How would you feel about a model that thinks for 60 seconds before answering? Is there a point where more thinking has diminishing returns?

## Part 3: GRPO — how reasoning is trained

Now let's implement a simplified version of **Group Relative Policy Optimization (GRPO)** — the algorithm DeepSeek used to train R1. We'll train a tiny model to solve simple arithmetic by discovering its own "reasoning" strategies.

### The setup

We'll train a small neural network to solve addition problems. Instead of supervised learning (showing it the steps), we'll use RL: generate multiple candidate answers, check which are correct, and reinforce the patterns that worked.

In [ ]:
class SimpleReasoningModel(nn.Module):
    """A tiny model that learns to solve addition via RL.
    
    Instead of directly predicting the answer, the model generates
    a sequence of 'intermediate tokens' (reasoning steps) before
    producing a final answer. The RL training signal comes only
    from whether the final answer is correct.
    """
    def __init__(self, max_val=100, hidden_dim=128, num_reasoning_steps=5):
        super().__init__()
        self.max_val = max_val
        self.num_reasoning_steps = num_reasoning_steps
        
        # Encode the two input numbers
        self.input_encoder = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.ReLU(),
        )
        
        # Reasoning steps: each step transforms the hidden state
        # This is analogous to generating thinking tokens
        self.reasoning_steps = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
            ) for _ in range(num_reasoning_steps)
        ])
        
        # Gate: decides how much each reasoning step contributes
        # (analogous to the model deciding whether to keep thinking)
        self.gates = nn.ModuleList([
            nn.Linear(hidden_dim, 1) for _ in range(num_reasoning_steps)
        ])
        
        # Output: predict the answer
        self.output_head = nn.Linear(hidden_dim, 2 * max_val + 1)  # possible sums: 0 to 2*max_val
    
    def forward(self, a, b):
        # Encode inputs
        x = torch.stack([a.float(), b.float()], dim=-1)
        h = self.input_encoder(x)
        
        # Apply reasoning steps with gating
        gate_values = []
        for step, gate in zip(self.reasoning_steps, self.gates):
            residual = h
            h_new = step(h)
            g = torch.sigmoid(gate(h))  # how much to use this step
            h = g * h_new + (1 - g) * residual
            gate_values.append(g.mean().item())
        
        # Produce answer logits
        logits = self.output_head(h)
        return logits, gate_values


model = SimpleReasoningModel(max_val=50, hidden_dim=64, num_reasoning_steps=5)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Reasoning steps: {model.num_reasoning_steps}")

In [ ]:
def grpo_train_step(model, batch_size=64, group_size=8, max_val=50, temperature=1.0):
    """One step of simplified GRPO training.
    
    1. Generate random addition problems
    2. For each problem, sample `group_size` candidate answers
    3. Check which are correct
    4. Compute relative advantage (how much better than group average)
    5. Update model to increase probability of above-average solutions
    """
    # Generate random problems: a + b = ?
    a = torch.randint(0, max_val, (batch_size,))
    b = torch.randint(0, max_val, (batch_size,))
    correct = a + b  # ground truth
    
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    for _ in range(group_size):
        # Forward pass
        logits, _ = model(a, b)
        
        # Sample an answer from the model's distribution
        probs = F.softmax(logits / temperature, dim=-1)
        sampled = torch.multinomial(probs, 1).squeeze(-1)
        
        # Check correctness (the ONLY reward signal)
        is_correct = (sampled == correct).float()
        total_correct += is_correct.sum().item()
        total_samples += batch_size
        
        # GRPO: compute advantage relative to group
        # For simplicity: correct = +1, wrong = -1
        # In full GRPO, this would be relative to the group's mean reward
        advantage = is_correct * 2 - 1  # +1 for correct, -1 for wrong
        
        # Policy gradient loss: increase log-prob of correct answers
        log_probs = F.log_softmax(logits, dim=-1)
        selected_log_probs = log_probs.gather(1, sampled.unsqueeze(1)).squeeze(1)
        
        # Loss = -advantage * log_prob (REINFORCE-style)
        loss = -(advantage * selected_log_probs).mean()
        total_loss += loss
    
    # Average over group
    total_loss = total_loss / group_size
    accuracy = total_correct / total_samples
    
    return total_loss, accuracy


# Training loop
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
accuracies = []
losses = []

print("Training with GRPO (RL only — no supervised examples)...")
print("The model must DISCOVER how to add numbers through trial and error.\n")

for step in range(500):
    optimizer.zero_grad()
    loss, acc = grpo_train_step(model, batch_size=64, group_size=8, max_val=50)
    loss.backward()
    optimizer.step()
    
    accuracies.append(acc)
    losses.append(loss.item())
    
    if (step + 1) % 50 == 0:
        print(f"  Step {step+1:4d}: Loss = {loss.item():.4f}, Accuracy = {acc:.1%}")

print(f"\nFinal accuracy: {accuracies[-1]:.1%}")

In [ ]:
# Visualize training progress
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Smooth the curves
window = 20
smooth_acc = np.convolve(accuracies, np.ones(window)/window, mode='valid')
smooth_loss = np.convolve(losses, np.ones(window)/window, mode='valid')

ax1.plot(smooth_acc, color="#00b894", linewidth=2)
ax1.set_title("Accuracy (GRPO Training)", fontsize=14)
ax1.set_xlabel("Step")
ax1.set_ylabel("Accuracy")
ax1.set_ylim(0, 1)
ax1.grid(alpha=0.3)
ax1.axhline(y=1/101, color='gray', linestyle='--', alpha=0.5, label='Random chance')
ax1.legend()

ax2.plot(smooth_loss, color="#e17055", linewidth=2)
ax2.set_title("Loss (GRPO Training)", fontsize=14)
ax2.set_xlabel("Step")
ax2.set_ylabel("Loss")
ax2.grid(alpha=0.3)

plt.suptitle("Learning to Add via Reinforcement Learning (No Supervised Examples)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Examine the learned gate values — does the model use all reasoning steps?
test_a = torch.randint(0, 50, (100,))
test_b = torch.randint(0, 50, (100,))

with torch.no_grad():
    _, gate_values = model(test_a, test_b)

plt.figure(figsize=(8, 5))
plt.bar(range(1, len(gate_values) + 1), gate_values, color="#6c5ce7", alpha=0.8)
plt.title("Reasoning Step Gate Values (How Much Each Step Contributes)", fontsize=13)
plt.xlabel("Reasoning Step", fontsize=12)
plt.ylabel("Gate Value (0 = skip, 1 = fully use)", fontsize=12)
plt.ylim(0, 1)
plt.xticks(range(1, len(gate_values) + 1))
plt.grid(axis='y', alpha=0.3)
plt.show()

print("Gate values per reasoning step:")
for i, g in enumerate(gate_values):
    bar = '\u2588' * int(g * 30)
    print(f"  Step {i+1}: {g:.3f} {bar}")

### 💡 Discussion

- The model learned to add numbers **without ever being shown how** — it received only a correct/incorrect signal. How does this compare to how DeepSeek-R1-Zero learned to reason?
- Look at the gate values: does the model use all reasoning steps equally, or does it rely more on some than others?
- In the real DeepSeek-R1, emergent behaviors included self-verification and backtracking. Our toy model can't do this (it's a feed-forward network, not autoregressive). What architectural feature of LLMs makes self-correction possible?
- If we increased `num_reasoning_steps` from 5 to 50, would the model get better? When would more steps stop helping?

## Part 4: Comparing reasoning steps to no reasoning

Let's directly compare models with different numbers of reasoning steps to see the effect of "thinking longer."

In [ ]:
# Train models with different numbers of reasoning steps
step_counts = [0, 1, 3, 5, 10]
final_accuracies = {}
training_curves = {}

for n_steps in step_counts:
    print(f"Training model with {n_steps} reasoning steps...")
    m = SimpleReasoningModel(max_val=50, hidden_dim=64, num_reasoning_steps=max(n_steps, 1))
    
    # If 0 steps, disable the gates (force them to 0)
    if n_steps == 0:
        for gate in m.gates:
            gate.weight.data.fill_(-10)  # sigmoid(-10) ≈ 0
            gate.bias.data.fill_(-10)
            gate.weight.requires_grad = False
            gate.bias.requires_grad = False
    
    opt = torch.optim.Adam(m.parameters(), lr=0.001)
    accs = []
    
    for step in range(300):
        opt.zero_grad()
        loss, acc = grpo_train_step(m, batch_size=64, group_size=8, max_val=50)
        loss.backward()
        opt.step()
        accs.append(acc)
    
    final_accuracies[n_steps] = np.mean(accs[-20:])  # average of last 20 steps
    training_curves[n_steps] = accs
    print(f"  Final accuracy: {final_accuracies[n_steps]:.1%}")

print("\n" + "=" * 40)
print("Summary:")
for n, acc in final_accuracies.items():
    print(f"  {n:2d} reasoning steps: {acc:.1%}")

In [ ]:
# Visualize the comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Training curves
colors = ["#d63031", "#e17055", "#fdcb6e", "#00b894", "#0984e3"]
for (n_steps, accs), color in zip(training_curves.items(), colors):
    window = 20
    smooth = np.convolve(accs, np.ones(window)/window, mode='valid')
    ax1.plot(smooth, label=f"{n_steps} steps", color=color, linewidth=2)

ax1.set_title("Training Curves by Reasoning Depth", fontsize=14)
ax1.set_xlabel("Training Step")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 1)

# Final accuracy bar chart
ax2.bar([str(n) for n in step_counts], 
        [final_accuracies[n] for n in step_counts],
        color=colors, alpha=0.8)
ax2.set_title("Final Accuracy vs. Reasoning Steps", fontsize=14)
ax2.set_xlabel("Number of Reasoning Steps")
ax2.set_ylabel("Accuracy")
ax2.set_ylim(0, 1)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle("More Thinking = Better Performance (up to a point)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 💡 Discussion

- Is the relationship between reasoning steps and accuracy linear? Does it plateau?
- In real reasoning models, the "number of steps" isn't fixed — the model generates tokens until it decides to stop. How might this adaptive behavior be better than a fixed number of steps?
- Our toy model has the same hidden dimension regardless of reasoning steps. In a real transformer, each reasoning token gets the full model's computation. How does this change the scaling dynamics?

## Part 5: Budget forcing — the s1 "Wait" trick

[Muennighoff et al. (2025)](https://arxiv.org/abs/2501.19393) showed that you can force a model to think longer by simply appending "Wait" to its reasoning. Let's test this with Claude — we'll ask a hard problem and compare what happens when we force additional reflection.

In [ ]:
# Budget forcing experiment: compare short vs. forced-longer reasoning

hard_problem = """What is the remainder when 7^100 is divided by 13?
Think carefully and show your reasoning."""

forced_problem = """What is the remainder when 7^100 is divided by 13?
Think carefully and show your reasoning.
After your first attempt at a solution, stop and say "Wait, let me verify this" and check your work from scratch using a completely different method."""

print("=" * 70)
print("BUDGET FORCING EXPERIMENT")
print("=" * 70)

print("\n--- Standard (let the model decide how long to think) ---")
result_standard = ask_with_thinking(hard_problem, budget_tokens=4000)
print(f"Thinking: {result_standard['thinking_length']:,} chars")
print(f"Time: {result_standard['elapsed']:.1f}s")
print(f"Answer (first 300 chars): {result_standard['answer'][:300]}")

print("\n--- Forced reflection (\"Wait, let me verify\") ---")
result_forced = ask_with_thinking(forced_problem, budget_tokens=8000)
print(f"Thinking: {result_forced['thinking_length']:,} chars")
print(f"Time: {result_forced['elapsed']:.1f}s")
print(f"Answer (first 300 chars): {result_forced['answer'][:300]}")

print(f"\n--- Comparison ---")
print(f"Standard thinking length:  {result_standard['thinking_length']:,} chars")
print(f"Forced thinking length:    {result_forced['thinking_length']:,} chars")
print(f"Ratio: {result_forced['thinking_length'] / max(result_standard['thinking_length'], 1):.1f}x more thinking")

### 💡 Discussion

- Did the forced verification lead to a different (possibly more correct) answer?
- The s1 paper found that appending "Wait" improved accuracy by up to 27%. Why might forcing the model to continue reasoning help — even when the model "thought" it was done?
- This is related to **premature convergence**: the model settles on an answer too quickly. Is this a problem unique to AI, or do humans do this too?
- What are the risks of forcing a model to think longer? Could it *overthink* and make things worse?

## Summary

| Concept | What we did | Key insight |
|---------|-------------|-------------|
| **Chain-of-thought** | Compared standard vs. CoT prompting | Intermediate steps = more computation = better reasoning |
| **Extended thinking** | Observed Claude's visible reasoning traces | Models self-verify, backtrack, and try alternatives |
| **GRPO training** | Implemented simplified RL training | Reasoning strategies emerge from reward signals alone |
| **Reasoning depth** | Compared models with different step counts | More steps help, but with diminishing returns |
| **Budget forcing** | Tested the s1 "Wait" trick | Forcing continued reasoning can improve accuracy |

## Further exploration

1. **Temperature and reasoning**: Try varying the `temperature` parameter in the GRPO training loop. Higher temperature = more exploration. Does this affect what reasoning strategies the model discovers?

2. **Problem difficulty**: Increase `max_val` in the GRPO experiment. At what point does the model need more reasoning steps to maintain accuracy? Is there a predictable relationship?

3. **Transfer**: Train the GRPO model on addition, then test it on subtraction (without retraining). Does the "reasoning" transfer, or is it task-specific?

4. **Thinking budget optimization**: For the Claude extended thinking experiment, try finding the *minimum* budget that still produces a correct answer. This is the "compute-optimal" strategy from Snell et al. (2024).

5. **Compare models**: Run the same extended thinking experiment with different Claude models (Haiku vs. Sonnet). Does a smaller model benefit more from thinking, as Snell et al. predict?